In [2]:
import os

python_311 = r"C:\Users\user\AppData\Local\Programs\Python\Python311\python.exe"

os.environ['PYSPARK_DRIVER_PYTHON'] = python_311
os.environ['PYSPARK_PYTHON'] = python_311

spark_home_pattern = r"%SPARK_HOME%"
pythonpath_value = rf"{spark_home_pattern}\python\lib\pyspark.zip;{spark_home_pattern}\python\lib\py4j-0.10.9.9-src.zip"

os.environ['PYTHONPATH'] = os.path.expandvars(pythonpath_value)

print(f"PYSPARK_DRIVER_PYTHON: {os.environ.get('PYSPARK_DRIVER_PYTHON')}")
print(f"PYSPARK_PYTHON: {os.environ.get('PYSPARK_PYTHON')}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

--- 설정된 환경변수 ---
PYSPARK_DRIVER_PYTHON: C:\Users\user\AppData\Local\Programs\Python\Python311\python.exe
PYSPARK_PYTHON: C:\Users\user\AppData\Local\Programs\Python\Python311\python.exe
PYTHONPATH: C:\Spark\spark-4.1.1-bin-hadoop3\python\lib\pyspark.zip;C:\Spark\spark-4.1.1-bin-hadoop3\python\lib\py4j-0.10.9.9-src.zip


In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'PySpark Functions')
conf.set('spark.master', 'local[*]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [8]:
data_list = [("Ravi", "28", "1", "2002"),
            ("Abdul", "23", "5", "81"),
            ("John", "12", "12", "6"),
            ("Rosy", "7", "8", "63"),
            ("Abdul", "23", "5", "81")]

df = spark.createDataFrame(data_list).toDF('name', 'day', 'month', 'year')

df.show()

+-----+---+-----+----+
| name|day|month|year|
+-----+---+-----+----+
| Ravi| 28|    1|2002|
|Abdul| 23|    5|  81|
| John| 12|   12|   6|
| Rosy|  7|    8|  63|
|Abdul| 23|    5|  81|
+-----+---+-----+----+



In [10]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- day: string (nullable = true)
 |-- month: string (nullable = true)
 |-- year: string (nullable = true)



## Misc

In [ ]:
import pyspark.sql.functions as f
from pyspark.sql.types import IntegerType

df.withColumn('id', f.monotonically_increasing_id())\
    .withColumn('day', f.col('day').cast(IntegerType()))\
    .withColumn('month', f.col('month').cast(IntegerType()))\
    .withColumn('year', f.col('year').cast(IntegerType()))\
    .withColumn('year', f.when(f.col('year') < 27, f.col('year') + 2000)\
                .when(f.col('year') < 100, f.col('year') + 1900)\
                .otherwise(f.col('year')))\
    .withColumn('dob', f.expr("to_date(concat(day, '/', month, '/', year), 'd/M/y')"))\
    .drop('day', 'month', 'year')\
    .dropDuplicates(['name', 'dob'])\
    .sort(f.col('dob').desc()).show()

# .withColumn('dob', f.make_date(f.col('year'), f.col('month'), f.col('day')))\

+-----+------------+----------+
| name|          id|       dob|
+-----+------------+----------+
| John| 77309411328|2006-12-12|
| Ravi| 25769803776|2002-01-28|
|Abdul| 51539607552|1981-05-23|
| Rosy|103079215104|1963-08-07|
+-----+------------+----------+



## Agg

data source: https://github.com/LearningJournal/Spark-Programming-In-Python/blob/master/13-AggDemo/data/invoices.csv

In [18]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType

schema = StructType([
    StructField('InvoiceNo', IntegerType(), True),
    StructField('StockCode', IntegerType(), True),
    StructField('Description', StringType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('InvoiceDate', StringType(), True),
    StructField('UnitPrice', FloatType(), True),
    StructField('CustomerID', IntegerType(), True),
    StructField('Country', StringType(), True)
])

df = spark.read.csv('invoices.csv', header = True, schema = schema)

df.printSchema()

df.show(5)

root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: integer (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   536365|     NULL|WHITE HANGING HEA...|       6|01-12-2010 8.26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|
|   536365|     NULL|CREAM CUPID HEART...|       8|01-12-2010 8.26|     2.75|     17850|United Kingdom|
|   536365|     NULL|KNITTED UNION FL

1. Spark DataFrame API

In [22]:
df.select(f.count('*').alias('Count'),
          f.sum('Quantity').alias('TotalQuantity'),
          f.avg('UnitPrice').alias('AvgPrice'),
          f.countDistinct('InvoiceNo').alias('CountDistinct')).show()

+------+-------------+-----------------+-------------+
| Count|TotalQuantity|         AvgPrice|CountDistinct|
+------+-------------+-----------------+-------------+
|541909|      5176450|4.611113614622465|        22061|
+------+-------------+-----------------+-------------+



In [47]:
df.groupBy('Country', 'InvoiceNo')\
    .agg(f.sum('Quantity').alias('TotalQuantity'),
         f.round(f.sum(f.expr('Quantity * UnitPrice')), 2).alias('InvoicePrice'),
         f.expr('round(sum(Quantity * UnitPrice), 2) as InvoicePrice')).show(5)

+--------------+---------+-------------+------------+------------+
|       Country|InvoiceNo|TotalQuantity|InvoicePrice|InvoicePrice|
+--------------+---------+-------------+------------+------------+
|United Kingdom|   536571|          284|      294.62|      294.62|
|        France|   537463|          585|     1033.52|     1033.52|
|        France|   539113|            3|        4.95|        4.95|
|United Kingdom|   539280|           17|       76.55|       76.55|
|United Kingdom|   540468|          948|     2311.53|     2311.53|
+--------------+---------+-------------+------------+------------+
only showing top 5 rows


2. SQL Expression

In [ ]:
df.selectExpr(
    'count(1) as `Count`',
    'count(StockCode) as `Count Field`',
    'sum(Quantity) as TotalQuantity',
    'avg(UnitPrice) as AvgPrice'
).show()

+------+-----------+-------------+-----------------+
| Count|Count Field|TotalQuantity|         AvgPrice|
+------+-----------+-------------+-----------------+
|541909|     487036|      5176450|4.611113614622466|
+------+-----------+-------------+-----------------+



3. Spark SQL

In [44]:
df.createOrReplaceTempView('df')

spark.sql("""
    SELECT Country, InvoiceNo,
          SUM(Quantity) AS TotalQuantity,
          ROUND(SUM(Quantity * UnitPrice), 2) as InvoiceValue
    FROM df
    GROUP BY 1, 2
""").show(5)

+--------------+---------+-------------+------------+
|       Country|InvoiceNo|TotalQuantity|InvoiceValue|
+--------------+---------+-------------+------------+
|United Kingdom|   536571|          284|      294.62|
|        France|   537463|          585|     1033.52|
|        France|   539113|            3|        4.95|
|United Kingdom|   539280|           17|       76.55|
|United Kingdom|   540468|          948|     2311.53|
+--------------+---------+-------------+------------+
only showing top 5 rows


## Grouping

In [71]:
NumInvoices = f.countDistinct('InvoiceNo').alias('NumInvoices')
TotalQuantity = f.sum('Quantity').alias('TotalQuantity')
InvoiceValue = f.expr('round(sum(Quantity * UnitPrice), 2) as InvoiceValue')

summary_df = df.withColumn('InvoiceDate', f.to_date(f.col('InvoiceDate'), 'dd-MM-yyyy H.mm'))\
                .where('year(InvoiceDate) == 2010')\
                .withColumn('WeekNumber', f.weekofyear(f.col('InvoiceDate')))\
                .groupBy('Country', 'WeekNumber')\
                .agg(NumInvoices, TotalQuantity, InvoiceValue)

summary_df.sort('Country', 'WeekNumber').show(5)

+---------+----------+-----------+-------------+------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|
+---------+----------+-----------+-------------+------------+
|Australia|        48|          1|          107|      358.25|
|Australia|        49|          1|          214|       258.9|
|Australia|        50|          1|          133|      387.95|
|  Austria|        50|          1|            3|      257.04|
|  Bahrain|        51|          1|           54|      205.74|
+---------+----------+-----------+-------------+------------+
only showing top 5 rows


## Window

In [74]:
df = spark.read.parquet('summary.parquet')

df.printSchema()
df.show(5)

root
 |-- Country: string (nullable = true)
 |-- WeekNumber: integer (nullable = true)
 |-- NumInvoices: long (nullable = true)
 |-- TotalQuantity: long (nullable = true)
 |-- InvoiceValue: double (nullable = true)

+---------+----------+-----------+-------------+------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|
+---------+----------+-----------+-------------+------------+
|    Spain|        49|          1|           67|      174.72|
|  Germany|        48|         11|         1795|     3309.75|
|Lithuania|        48|          3|          622|     1598.06|
|  Germany|        49|         12|         1852|     4521.39|
|  Bahrain|        51|          1|           54|      205.74|
+---------+----------+-----------+-------------+------------+
only showing top 5 rows


In [76]:
from pyspark.sql import Window

df.withColumn('RunningTotal', f.sum('Invoicevalue').over(Window.partitionBy('Country')\
                                                    .orderBy('WeekNumber')\
                                                    .rowsBetween(-2, Window.currentRow))).show(10)

+---------------+----------+-----------+-------------+------------+------------------+
|        Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|      RunningTotal|
+---------------+----------+-----------+-------------+------------+------------------+
|      Australia|        48|          1|          107|      358.25|            358.25|
|      Australia|        49|          1|          214|       258.9|            617.15|
|      Australia|        50|          2|          133|      387.95|1005.0999999999999|
|        Austria|        50|          2|            3|      257.04|            257.04|
|        Bahrain|        51|          1|           54|      205.74|            205.74|
|        Belgium|        48|          1|          528|       346.1|             346.1|
|        Belgium|        50|          2|          285|      625.16|            971.26|
|        Belgium|        51|          2|          942|      838.65|1809.9099999999999|
|Channel Islands|        49|          1|   

In [77]:
summary_df = spark.read.parquet('summary.parquet')

summary_df.sort('Country', 'WeekNumber').show(5)

+---------+----------+-----------+-------------+------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|
+---------+----------+-----------+-------------+------------+
|Australia|        48|          1|          107|      358.25|
|Australia|        49|          1|          214|       258.9|
|Australia|        50|          2|          133|      387.95|
|  Austria|        50|          2|            3|      257.04|
|  Bahrain|        51|          1|           54|      205.74|
+---------+----------+-----------+-------------+------------+
only showing top 5 rows


In [90]:
summary_df.withColumn('Rank', f.row_number().over(Window.partitionBy('Country')\
                                                  .orderBy(f.col('InvoiceValue').desc())\
                                                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)))\
        .where(f.col('Rank') == 1)\
        .sort('Country', 'WeekNumber').show(10)

+---------------+----------+-----------+-------------+------------+----+
|        Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|Rank|
+---------------+----------+-----------+-------------+------------+----+
|      Australia|        50|          2|          133|      387.95|   1|
|        Austria|        50|          2|            3|      257.04|   1|
|        Bahrain|        51|          1|           54|      205.74|   1|
|        Belgium|        51|          2|          942|      838.65|   1|
|Channel Islands|        49|          1|           80|      363.53|   1|
|         Cyprus|        50|          1|          917|     1590.82|   1|
|        Denmark|        49|          1|          454|      1281.5|   1|
|           EIRE|        49|          5|         1280|      3284.1|   1|
|        Finland|        50|          1|         1254|       892.8|   1|
|         France|        49|          9|         2303|     4527.01|   1|
+---------------+----------+-----------+-----------